In [ ]:
# Necessary Imports
import os
import gc
import time
import sys

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import numpy as np
import h5py # For load_mat_data
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, cohen_kappa_score

import matplotlib.pyplot as plt
%matplotlib inline

# --- Configuration ---
MAT_FILE_PATH = "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38.mat" #  MODIFIED: Update with your MAT file path
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RANDOM_STATE = 42
NUM_EPOCHS_DEMO = 5 #  Keep epochs low for quick comparison
BATCH_SIZE = 256 #  Adjust based on your GPU memory

# Assuming 102 raw dimensions in one-hot labels, where 0 is BG
# When BG is processed, we have 101 actual classes (0-100)
NUM_ACTUAL_CLASSES = 101
# When BG is NOT processed, we have 102 classes (0-101, where 0 is BG)
NUM_TOTAL_CLASSES_WITH_BG = 102

# --- 1. MAT Data Loading Utilities (adapted from brain_voxel_comparison.py) ---
def load_mat_data_from_h5(mat_file_path):
    """Simplified MAT loader from h5py."""
    print(f"Loading MAT data from: {mat_file_path}")
    arrays = {}
    with h5py.File(mat_file_path, 'r') as f:
        for k, v in f.items():
            arrays[k] = np.array(v)
    print("MAT data loaded.")
    return arrays

def check_and_transpose_data_fn(arrays):
    """Checks and transposes MAT data (from brain_voxel_comparison.py)."""
    print("Checking and transposing data...")
    data_key, region_key, prob_idx_key = 'data', 'region', 'prob_idx'

    if data_key not in arrays or region_key not in arrays or prob_idx_key not in arrays:
        # Try to find keys if they have prefixes like 'TRAIN_DATA', 'TRAIN_REGION'
        # This is a common pattern in some .mat workspace saves.
        # For simplicity, this example assumes direct keys.
        # In a real scenario, you might need more robust key discovery.
        print("Error: Standard keys 'data', 'region', 'prob_idx' not found.")
        # Fallback: list keys and ask user or use first compatible
        print(f"Available keys: {list(arrays.keys())}")
        # Example: If keys are like 'TD', 'TR', 'TP'
        # data_key = 'TD'; region_key = 'TR'; prob_idx_key = 'TP' # Adjust as needed
        raise KeyError("Could not find standard data keys in .mat file.")


    # Data: (Features, Samples) -> (Samples, Features)
    if arrays[data_key].shape[0] == 341: # Assuming 341 features
        data_transposed = arrays[data_key].T
    elif arrays[data_key].shape[1] == 341:
        data_transposed = arrays[data_key]
    else:
        raise ValueError(f"Unexpected data shape: {arrays[data_key].shape}")

    # Region: (Classes, Samples) -> (Samples, Classes)
    if arrays[region_key].shape[0] == NUM_TOTAL_CLASSES_WITH_BG: # 102 classes including BG
        region_transposed = arrays[region_key].T
    elif arrays[region_key].shape[1] == NUM_TOTAL_CLASSES_WITH_BG:
        region_transposed = arrays[region_key]
    else:
        raise ValueError(f"Unexpected region shape: {arrays[region_key].shape}")

    prob_idx_transposed = arrays[prob_idx_key].flatten()

    assert data_transposed.shape[0] == region_transposed.shape[0] == prob_idx_transposed.shape[0], "Sample count mismatch"
    assert data_transposed.shape[1] == 341, "Feature count mismatch"
    assert region_transposed.shape[1] == NUM_TOTAL_CLASSES_WITH_BG, "Class count mismatch"
    print("Data transposed and validated.")
    return data_transposed, region_transposed, prob_idx_transposed

def split_data_fn(data, region, prob_idx, random_state=RANDOM_STATE):
    """Splits data based on patient IDs."""
    print("Splitting data by patient ID...")
    # Patient 38 for validation
    val_mask = (prob_idx == 38)
    val_indices = np.where(val_mask)[0]

    # Others for train/test pool
    train_test_pool_mask = ~val_mask
    train_test_pool_indices = np.where(train_test_pool_mask)[0]

    # Split train_test_pool (1% for test)
    if len(train_test_pool_indices) > 0 :
        train_indices, test_indices = train_test_split(
            train_test_pool_indices, test_size=0.01, random_state=random_state
        )
    else: # Handle cases with very few patients if P38 is the only one or similar
        train_indices = np.array([], dtype=int)
        test_indices = np.array([], dtype=int)

    print(f"Train samples: {len(train_indices)}, Val samples: {len(val_indices)}, Test samples: {len(test_indices)}")
    return train_indices, val_indices, test_indices

# --- 2. Scaler Creation (adapted from brain_voxel_comparison.py) ---
def create_scaler_fn(all_data, train_indices):
    """Creates and fits StandardScaler on training data in batches."""
    print("Creating and fitting StandardScaler on training data...")
    scaler = StandardScaler()
    
    # Fit scaler in batches to save memory (if necessary, for very large data)
    # For this demo, if train_indices is not excessively large, fit directly
    # For very large datasets, the batch-wise mean/std calculation from
    # brain_voxel_comparison.py (create_scaler_from_full_training_set) is better.
    # Here, a simpler direct fit for demonstration if memory allows.
    if len(train_indices) == 0:
        print("Warning: No training data to fit scaler.")
        # Return an unfitted scaler or handle as an error
        return scaler

    print(f"Fitting scaler on {len(train_indices)} training samples.")
    scaler.fit(all_data[train_indices])
    print("StandardScaler fitted.")
    return scaler

# --- 3. Custom PyTorch Dataset ---
class BrainVoxelPyTorchDataset(Dataset):
    def __init__(self, all_features, all_one_hot_labels, indices, scaler,
                 process_background=True):
        self.features = all_features[indices] # Select only relevant samples
        self.one_hot_labels = all_one_hot_labels[indices]
        self.scaler = scaler
        self.process_background = process_background

        # Scale features
        self.features = self.scaler.transform(self.features)

        # Process labels
        indexed_raw_labels = np.argmax(self.one_hot_labels, axis=1) # Values 0 to 101

        if self.process_background:
            # BG (original index 0) -> -1 (ignore)
            # Actual classes (original indices 1-101) -> 0-100
            self.processed_labels = indexed_raw_labels - 1
            self.processed_labels[indexed_raw_labels == 0] = -1 # Set original BG to -1
        else:
            # BG is class 0, actual classes are 1-101
            self.processed_labels = indexed_raw_labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        features = torch.FloatTensor(self.features[idx])
        labels = torch.LongTensor([self.processed_labels[idx]]).squeeze() # Ensure scalar tensor
        return features, labels

# --- 4. Model Definition (FixedMLP from brain_voxel_comparison.py) ---
class FixedMLP(nn.Module):
    def __init__(self, input_dim=341, num_output_classes=101, dropout_rate=0.5, l2_reg=1e-5):
        super(FixedMLP, self).__init__()
        self.l2_reg = l2_reg
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 4096), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(4096, 4096), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(4096, 4096), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(4096, 4096), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(4096, num_output_classes) # Output layer size changes
        )
        # Weight initialization (optional, but good practice)
        for layer in self.layers:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)
    def forward(self, x):
        return self.layers(x)

    def get_l2_loss(self): # For manual L2
        l2_loss = 0
        for param in self.parameters():
            if param.requires_grad: # Usually all model params do
                 l2_loss += torch.norm(param, 2) ** 2
        return self.l2_reg * l2_loss


# --- 5. Class Weight Calculation ---
def calculate_weights_fn(labels_for_weights, num_effective_classes, process_background):
    """Calculates class weights."""
    print(f"Calculating class weights for {num_effective_classes} effective classes.")
    if process_background: # Labels are -1 (BG), 0 to N-1 (actual classes)
        counts = np.bincount(labels_for_weights[labels_for_weights >= 0], minlength=num_effective_classes)
    else: # Labels are 0 (BG), 1 to N (actual classes)
        counts = np.bincount(labels_for_weights, minlength=num_effective_classes)

    weights = np.zeros(num_effective_classes, dtype=np.float32)
    total_valid_samples = np.sum(counts)
    
    # smoothed weighting: 1 / (count + epsilon) then normalize, or total_samples / (num_classes * count)
    # Using the common total_samples / (num_classes * count)
    for i in range(num_effective_classes):
        if counts[i] > 0:
            weights[i] = total_valid_samples / (num_effective_classes * counts[i])
        else:
            weights[i] = 0 # Or a small default if you want to avoid zero weight for unseen train classes
    
    # If some classes have 0 count, their weight is 0.
    # It might be better to assign a default high weight or handle this.
    # For simplicity, we'll use 0 for now, CrossEntropyLoss handles 0 weight correctly.
    if np.any(weights == 0):
        print(f"Warning: {np.sum(weights==0)} classes have zero weight due to no samples in the provided labels.")

    print(f"Class weights calculated. Min_weight (for present classes): {np.min(weights[weights>0]):.4f}, Max_weight: {np.max(weights):.4f}")
    return torch.FloatTensor(weights).to(DEVICE)


# --- 6. Training and Evaluation Loops ---
def train_epoch_fn(model, dataloader, criterion, optimizer, device, add_l2_loss=True):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_true = []

    for features, labels in dataloader:
        features, labels = features.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(features)
        
        classification_loss = criterion(outputs, labels)
        
        if add_l2_loss and hasattr(model, 'get_l2_loss'): # Check if model has L2
            l2_loss = model.get_l2_loss()
            loss = classification_loss + l2_loss
        else:
            loss = classification_loss

        loss.backward()
        optimizer.step()
        running_loss += loss.item() * features.size(0)

        # For metrics, store predictions and true labels
        # Only consider non-ignored labels for metrics if ignore_index is used by criterion
        if criterion.ignore_index is not None:
            valid_mask = (labels != criterion.ignore_index)
            if valid_mask.sum() > 0:
                preds = torch.argmax(outputs[valid_mask], dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_true.extend(labels[valid_mask].cpu().numpy())
        else:
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_true.extend(labels.cpu().numpy())
            
    epoch_loss = running_loss / len(dataloader.dataset)
    # Calculate metrics only if there are valid samples
    epoch_acc = accuracy_score(all_true, all_preds) if len(all_true) > 0 else 0.0
    epoch_f1 = f1_score(all_true, all_preds, average='macro', zero_division=0) if len(all_true) > 0 else 0.0
    
    return epoch_loss, epoch_acc, epoch_f1


def evaluate_model_fn(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_true = []

    with torch.no_grad():
        for features, labels in dataloader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            loss = criterion(outputs, labels) # Note: L2 is not typically added to validation loss
            running_loss += loss.item() * features.size(0)

            if criterion.ignore_index is not None:
                valid_mask = (labels != criterion.ignore_index)
                if valid_mask.sum() > 0:
                    preds = torch.argmax(outputs[valid_mask], dim=1)
                    all_preds.extend(preds.cpu().numpy())
                    all_true.extend(labels[valid_mask].cpu().numpy())
            else:
                preds = torch.argmax(outputs, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_true.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_true, all_preds) if len(all_true) > 0 else 0.0
    epoch_f1 = f1_score(all_true, all_preds, average='macro', zero_division=0) if len(all_true) > 0 else 0.0
    
    return epoch_loss, epoch_acc, epoch_f1

# --- Load and Prepare Full Dataset (once) ---
print("--- Initial Data Loading and Preparation ---")
mat_arrays = load_mat_data_from_h5(MAT_FILE_PATH)
all_features_raw, all_labels_one_hot_raw, prob_idx_raw = check_and_transpose_data_fn(mat_arrays)
train_idx, val_idx, test_idx = split_data_fn(all_features_raw, all_labels_one_hot_raw, prob_idx_raw)

# Create scaler based on training data
# Important: Ensure train_idx is not empty
if len(train_idx) == 0:
    raise ValueError("No training samples after splitting. Cannot proceed.")
    
scaler = create_scaler_fn(all_features_raw, train_idx)
print(f"Scaler mean (first 5 features): {scaler.mean_[:5]}")

# Store results
results_summary = {}
# 在运行所有场景前
initial_scaler_mean = scaler.mean_[:5].copy()
print(f"初始scaler均值: {initial_scaler_mean}")

In [ ]:

# --- Run the Four Scenarios ---
print(f"场景1开始前scaler均值: {scaler.mean_[:5]}")
assert np.allclose(scaler.mean_[:5], initial_scaler_mean), "Scaler状态已改变!"


# Scenario 1: WITH Background Processing, WITH Class Weights
print("\n\n--- Scenario 1: WITH Background Processing, WITH Class Weights ---")
# Data
train_dataset_s1 = BrainVoxelPyTorchDataset(all_features_raw, all_labels_one_hot_raw, train_idx, scaler, process_background=True)
val_dataset_s1 = BrainVoxelPyTorchDataset(all_features_raw, all_labels_one_hot_raw, val_idx, scaler, process_background=True)
train_loader_s1 = DataLoader(train_dataset_s1, batch_size=BATCH_SIZE, shuffle=True)
val_loader_s1 = DataLoader(val_dataset_s1, batch_size=BATCH_SIZE, shuffle=False)
# Model
model_s1 = FixedMLP(num_output_classes=NUM_ACTUAL_CLASSES).to(DEVICE) # 101 classes
# Weights & Criterion
# Use labels from the dataset object for weight calculation as they are already processed
weights_s1 = calculate_weights_fn(train_dataset_s1.processed_labels, NUM_ACTUAL_CLASSES, process_background=True)
criterion_s1 = nn.CrossEntropyLoss(weight=weights_s1, ignore_index=-1)
optimizer_s1 = optim.Adam(model_s1.parameters(), lr=1e-5) # L2 already in model's loss

history_s1 = {'train_loss':[], 'train_acc':[], 'train_f1':[], 'val_loss':[], 'val_acc':[], 'val_f1':[]}
for epoch in range(NUM_EPOCHS_DEMO):
    train_loss, train_acc, train_f1 = train_epoch_fn(model_s1, train_loader_s1, criterion_s1, optimizer_s1, DEVICE)
    val_loss, val_acc, val_f1 = evaluate_model_fn(model_s1, val_loader_s1, criterion_s1, DEVICE)
    history_s1['train_loss'].append(train_loss); history_s1['train_acc'].append(train_acc); history_s1['train_f1'].append(train_f1)
    history_s1['val_loss'].append(val_loss); history_s1['val_acc'].append(val_acc); history_s1['val_f1'].append(val_f1)
    print(f"S1 - Epoch {epoch+1}/{NUM_EPOCHS_DEMO} -> Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f} | Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}")
results_summary['Scenario 1 (With BG, With Weights)'] = {'val_acc': val_acc, 'val_f1': val_f1, 'history': history_s1}
del model_s1, train_dataset_s1, val_dataset_s1, train_loader_s1, val_loader_s1, weights_s1, criterion_s1, optimizer_s1; gc.collect(); torch.cuda.empty_cache()

print(f"场景1结束后scaler均值: {scaler.mean_[:5]}")

In [ ]:

# Scenario 2: WITHOUT Background Processing, WITH Class Weights
print(f"场景2开始前scaler均值: {scaler.mean_[:5]}")
assert np.allclose(scaler.mean_[:5], initial_scaler_mean), "Scaler状态已改变!"


print("\n\n--- Scenario 2: WITHOUT Background Processing, WITH Class Weights ---")
# Data
train_dataset_s2 = BrainVoxelPyTorchDataset(all_features_raw, all_labels_one_hot_raw, train_idx, scaler, process_background=False)
val_dataset_s2 = BrainVoxelPyTorchDataset(all_features_raw, all_labels_one_hot_raw, val_idx, scaler, process_background=False)
train_loader_s2 = DataLoader(train_dataset_s2, batch_size=BATCH_SIZE, shuffle=True)
val_loader_s2 = DataLoader(val_dataset_s2, batch_size=BATCH_SIZE, shuffle=False)
# Model
model_s2 = FixedMLP(num_output_classes=NUM_TOTAL_CLASSES_WITH_BG).to(DEVICE) # 102 classes
# Weights & Criterion
weights_s2 = calculate_weights_fn(train_dataset_s2.processed_labels, NUM_TOTAL_CLASSES_WITH_BG, process_background=False)
criterion_s2 = nn.CrossEntropyLoss(weight=weights_s2) # No ignore_index
optimizer_s2 = optim.Adam(model_s2.parameters(), lr=1e-5)

history_s2 = {'train_loss':[], 'train_acc':[], 'train_f1':[], 'val_loss':[], 'val_acc':[], 'val_f1':[]}
for epoch in range(NUM_EPOCHS_DEMO):
    train_loss, train_acc, train_f1 = train_epoch_fn(model_s2, train_loader_s2, criterion_s2, optimizer_s2, DEVICE)
    val_loss, val_acc, val_f1 = evaluate_model_fn(model_s2, val_loader_s2, criterion_s2, DEVICE)
    history_s2['train_loss'].append(train_loss); history_s2['train_acc'].append(train_acc); history_s2['train_f1'].append(train_f1)
    history_s2['val_loss'].append(val_loss); history_s2['val_acc'].append(val_acc); history_s2['val_f1'].append(val_f1)
    print(f"S2 - Epoch {epoch+1}/{NUM_EPOCHS_DEMO} -> Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f} | Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}")
results_summary['Scenario 2 (No BG, With Weights)'] = {'val_acc': val_acc, 'val_f1': val_f1, 'history': history_s2}
del model_s2, train_dataset_s2, val_dataset_s2, train_loader_s2, val_loader_s2, weights_s2, criterion_s2, optimizer_s2; gc.collect(); torch.cuda.empty_cache()

print(f"场景2结束后scaler均值: {scaler.mean_[:5]}")

In [ ]:

# Scenario 3: WITH Background Processing, WITHOUT Class Weights
print(f"场景3开始前scaler均值: {scaler.mean_[:5]}")
assert np.allclose(scaler.mean_[:5], initial_scaler_mean), "Scaler状态已改变!"


print("\n\n--- Scenario 3: WITH Background Processing, WITHOUT Class Weights ---")
# Data
train_dataset_s3 = BrainVoxelPyTorchDataset(all_features_raw, all_labels_one_hot_raw, train_idx, scaler, process_background=True)
val_dataset_s3 = BrainVoxelPyTorchDataset(all_features_raw, all_labels_one_hot_raw, val_idx, scaler, process_background=True)
train_loader_s3 = DataLoader(train_dataset_s3, batch_size=BATCH_SIZE, shuffle=True)
val_loader_s3 = DataLoader(val_dataset_s3, batch_size=BATCH_SIZE, shuffle=False)
# Model
model_s3 = FixedMLP(num_output_classes=NUM_ACTUAL_CLASSES).to(DEVICE) # 101 classes
# Criterion
criterion_s3 = nn.CrossEntropyLoss(ignore_index=-1) # No weights
optimizer_s3 = optim.Adam(model_s3.parameters(), lr=1e-5)

history_s3 = {'train_loss':[], 'train_acc':[], 'train_f1':[], 'val_loss':[], 'val_acc':[], 'val_f1':[]}
for epoch in range(NUM_EPOCHS_DEMO):
    train_loss, train_acc, train_f1 = train_epoch_fn(model_s3, train_loader_s3, criterion_s3, optimizer_s3, DEVICE)
    val_loss, val_acc, val_f1 = evaluate_model_fn(model_s3, val_loader_s3, criterion_s3, DEVICE)
    history_s3['train_loss'].append(train_loss); history_s3['train_acc'].append(train_acc); history_s3['train_f1'].append(train_f1)
    history_s3['val_loss'].append(val_loss); history_s3['val_acc'].append(val_acc); history_s3['val_f1'].append(val_f1)
    print(f"S3 - Epoch {epoch+1}/{NUM_EPOCHS_DEMO} -> Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f} | Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}")
results_summary['Scenario 3 (With BG, No Weights)'] = {'val_acc': val_acc, 'val_f1': val_f1, 'history': history_s3}
del model_s3, train_dataset_s3, val_dataset_s3, train_loader_s3, val_loader_s3, criterion_s3, optimizer_s3; gc.collect(); torch.cuda.empty_cache()

print(f"场景3结束后scaler均值: {scaler.mean_[:5]}")

In [ ]:

# Scenario 4: WITHOUT Background Processing, WITHOUT Class Weights
print(f"场景4开始前scaler均值: {scaler.mean_[:5]}")
assert np.allclose(scaler.mean_[:5], initial_scaler_mean), "Scaler状态已改变!"

print("\n\n--- Scenario 4: WITHOUT Background Processing, WITHOUT Class Weights ---")
# Data
train_dataset_s4 = BrainVoxelPyTorchDataset(all_features_raw, all_labels_one_hot_raw, train_idx, scaler, process_background=False)
val_dataset_s4 = BrainVoxelPyTorchDataset(all_features_raw, all_labels_one_hot_raw, val_idx, scaler, process_background=False)
train_loader_s4 = DataLoader(train_dataset_s4, batch_size=BATCH_SIZE, shuffle=True)
val_loader_s4 = DataLoader(val_dataset_s4, batch_size=BATCH_SIZE, shuffle=False)
# Model
model_s4 = FixedMLP(num_output_classes=NUM_TOTAL_CLASSES_WITH_BG).to(DEVICE) # 102 classes
# Criterion
criterion_s4 = nn.CrossEntropyLoss() # No ignore_index, No weights
optimizer_s4 = optim.Adam(model_s4.parameters(), lr=1e-5)

history_s4 = {'train_loss':[], 'train_acc':[], 'train_f1':[], 'val_loss':[], 'val_acc':[], 'val_f1':[]}
for epoch in range(NUM_EPOCHS_DEMO):
    train_loss, train_acc, train_f1 = train_epoch_fn(model_s4, train_loader_s4, criterion_s4, optimizer_s4, DEVICE)
    val_loss, val_acc, val_f1 = evaluate_model_fn(model_s4, val_loader_s4, criterion_s4, DEVICE)
    history_s4['train_loss'].append(train_loss); history_s4['train_acc'].append(train_acc); history_s4['train_f1'].append(train_f1)
    history_s4['val_loss'].append(val_loss); history_s4['val_acc'].append(val_acc); history_s4['val_f1'].append(val_f1)
    print(f"S4 - Epoch {epoch+1}/{NUM_EPOCHS_DEMO} -> Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f} | Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}")
results_summary['Scenario 4 (No BG, No Weights)'] = {'val_acc': val_acc, 'val_f1': val_f1, 'history': history_s4}
del model_s4, train_dataset_s4, val_dataset_s4, train_loader_s4, val_loader_s4, criterion_s4, optimizer_s4; gc.collect(); torch.cuda.empty_cache()

print(f"场景4结束后scaler均值: {scaler.mean_[:5]}")

In [ ]:

# --- 7. Results Comparison ---
print("\n\n--- Final Results Summary ---")
print(f"{'Scenario':<40} | {'Val Accuracy':<15} | {'Val Macro F1-Score':<20}")
print("-" * 80)
for scenario, metrics in results_summary.items():
    print(f"{scenario:<40} | {metrics['val_acc']:.4f}{' ':<10} | {metrics['val_f1']:.4f}")

# --- Plotting training curves (example for F1 score) ---
plt.figure(figsize=(12, 8))
for scenario_name, result in results_summary.items():
    plt.plot(result['history']['val_f1'], label=f"{scenario_name} - Val F1")
plt.title('Validation F1-Score Comparison Across Scenarios')
plt.xlabel(f'Epoch (Total {NUM_EPOCHS_DEMO})')
plt.ylabel('Macro F1-Score')
plt.legend(loc='best', bbox_to_anchor=(1.05, 1)) # Move legend out
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 8))
for scenario_name, result in results_summary.items():
    plt.plot(result['history']['val_acc'], label=f"{scenario_name} - Val Accuracy")
plt.title('Validation Accuracy Comparison Across Scenarios')
plt.xlabel(f'Epoch (Total {NUM_EPOCHS_DEMO})')
plt.ylabel('Accuracy')
plt.legend(loc='best', bbox_to_anchor=(1.05, 1))
plt.grid(True)
plt.tight_layout()